### Script irá ataur para criar uma tabela unica contendo os processos cíveis por canais de atendimento ###

In [0]:
%run "/Workspace/Jurídico/Normalizador_de_dados/Normalizador"  

In [0]:
base_on = f'/Volumes/databox/juridico_comum/arquivos/cível/Civil_processos_x_canal/Base_Canal/Base_atendimento_on.xlsx'
base_off = f'/Volumes/databox/juridico_comum/arquivos/cível/Civil_processos_x_canal/Base_Canal/Base_atendimentos_off.xlsx'
base_civil_ativos = f'/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Civel/Bases_Mensais_Ativo_Encerrado/CIVEL_GERENCIAL_(ATIVOS) - 20260211.xlsx'
base_civil_encerrados = f'/Volumes/databox/juridico_comum/arquivos/financeiro/Faturamento_Civel/Bases_Mensais_Ativo_Encerrado/CIVEL_GERENCIAL_(ENCERRADOS)-20260211.xlsx'
base_produto24_jur = f'/Volumes/databox/juridico_comum/arquivos/cível/Civil_processos_x_canal/Base_juridico/INFORMAÇÕES DE PRODUTO DE COMPRA -  DATA BASE 2024.xlsx'
base_produto25_jur = f'/Volumes/databox/juridico_comum/arquivos/cível/Civil_processos_x_canal/Base_juridico/INFORMAÇÕES DE PRODUTO DE COMPRA - DATA BASE JAN 2025 A 2026 (2).xlsx'


In [0]:
df_ativos = pd.read_excel(
    base_civil_ativos,
    sheet_name='CÍVEL',
    header=5 
)
df_encerrados = pd.read_excel(
    base_civil_encerrados,
    sheet_name='CÍVEL',
    header=5
)
df_on = pd.read_excel(
    base_on,
    sheet_name='Export',
    header=0
)
df_off = pd.read_excel(
    base_off,
    sheet_name='Export',
    header=0
)

df_24 = pd.read_excel(
    base_produto24_jur,
    sheet_name='INFORMAÇÕES DOS PRODUTOS DE COM',
    header=5
)
df_25 = pd.read_excel(
    base_produto25_jur,
    sheet_name='INFORMAÇÕES DOS PRODUTOS DE COM',
    header=5
)


In [0]:

df_concatenado_ger = pd.concat([df_ativos, df_encerrados], axis=0)
df_24_25 = pd.concat([df_24, df_25], axis=0)

In [0]:
regra_canal = df_concatenado_ger['COMPRA DIRETA OU MARKETPLACE? - NEW '].isin(['DIRETA', 'MARKETPLACE'])

# 2. Define a regra da empresa
regra_empresa = df_concatenado_ger['EMPRESA'].isin(['CASAS BAHIA ONLINE', 'GRUPO CASAS BAHIA S/A'])

# 3. Aplica ambas
df_concatenado_ger = df_concatenado_ger[regra_canal & regra_empresa]

In [0]:
df_final = df_concatenado_ger

In [0]:
df_final['DATA REGISTRADO'] = pd.to_datetime(df_final['DATA REGISTRADO'])

df_ordenado = df_final.sort_values(by='DATA REGISTRADO', ascending=False)

df_final = df_ordenado.drop_duplicates(subset='PROCESSO - ID', keep='first')

df_final = df_final[df_final['DATA REGISTRADO'] >= '2024-01-01']

In [0]:
# 1. Cria um DataFrame temporário com TODAS as divisões possíveis
temp_split = df_final['Pedido'].str.split(',', expand=True)

# 2. Seleciona apenas as 3 primeiras colunas (se houver mais, ignora; se houver menos, dá erro na renomeação abaixo)
# O iloc garante que pegamos apenas as posições 0, 1 e 2
temp_split = temp_split.iloc[:, :3]

# 3. Define os nomes das colunas
# Se houver chance de ter MENOS de 3 colunas, você precisa garantir que existam 3 antes de renomear
# Este comando garante que existam 3 colunas, preenchendo com None se faltar
temp_split = temp_split.reindex(columns=[0, 1, 2]) 

temp_split.columns = ['Info_Pedido_1', 'Info_Pedido_2', 'Info_Pedido_3']

# 4. Junta tudo com o DataFrame original
df_final = pd.concat([df_final, temp_split], axis=1)

In [0]:
# O comando list força o Python a mostrar todos os itens
print(list(df_final.columns))

In [0]:
df_final = df_final.drop(columns=['PASTA', 'ESTEIRA', 'CENTRO DE CUSTO / ÁREA DEMANDANTE - CÓDIGO', 'ADVOGADO RESPONSÁVEL', 'FORO/TRIBUNAL/ÓRGÃO', 'VARA/ÓRGÃO', 'AÇÃO', 'DISTRIBUIÇÃO', 'PARTE CONTRÁRIA CPF', 'ESCRITÓRIO EXTERNO', 'Escritório Anterior', 'FASE', 'USUÁRIO CADASTRANTE', 'DATA DA BAIXA PROVISÓRIA', 'DATA DA REATIVAÇÃO', 'VALOR PROVISIONADO', 'VALOR PROVISIONADO ATUALIZADO', 'PROCESSO CRITICO? (CÍVEL)', 'ASSISTENTE JURÍDICO - CÍVEL', 'ASSISTENTE JURÍDICO -', 'DATA AUDIENCIA INICIAL', 'RECLAMAÇÃO BLACK FRIDAY?', 'OUTRAS PARTES / NÃO-CLIENTES', 'MOTIVO DA CRITICIDADE', 'ADVOGADO DA PARTE CONTRÁRIA', 'DATA DE REABERTURA', 'DATA DE RECEBIMENTO', 'PARTE DESCRITA NA INICIAL', 'TIPO DE CONTINGÊNCIA', 'ANO DA BLACK FRIDAY', 'DATA DA ÚLTIMA MOVIMENTAÇÃO PROCESSUAL', 'OBSERVAÇÕES', 'OBSERVAÇÃO', 'TERCEIRO SOLVENTE', 'INCONTROVERSO (ZERAR PROVISÃO)', 'RESULTADO', 'DATA DE SOLICITAÇÃO DE ENCERRAMENTO', 'DATA DE ENCERRAMENTO NO TRIBUNAL', 'MOTIVO DE ENCERRAMENTO ', 'CÍVEL MASSA - RESPONSABILIDADE', 'ESTRATÉGIA - DEFINIR ESTRATÉGIA DE ATUAÇÃO E JUSTIFICATIVA', 'JUSTIFICATIVA - DEFINIR ESTRATÉGIA DE ATUAÇÃO E JUSTIFICATIVA', 'CÍVEL MASSA - TIPOS DE TERCEIRO', 'PARTE CONTRÁRIA DATA ADMISSÃO', 'PARTE CONTRÁRIA DATA DISPENSA', 'PARTE CONTRÁRIA - STATUS EM FOLHA', 'PARTE CONTRÁRIA - MATRICULA', 'PARTE CONTRÁRIA - CARGO - CARGO (GRUPO)', 'MOTIVO DA BAIXA PROVISÓRIA','LISTA LOJISTA - NOME', 'LISTA LOJISTA - ID', 'TRANSPORTADORA', 'NÚMERO DA NOTA FISCAL ', 'SITE DA COMPRA', 'DATA DO FATO GERADOR', 'DATA FATO GERADOR', 'DATA DO FATO GERADOR.1', 'DATA DO FATO GERADOR.2', 'Data da Baixa Provisória', 'DATA DE REATIVAÇÃO - nativo', 'DATA DA REATIVAÇÃO - ', 'E-MAIL - PARTE CONTRÁRIA', 'Processo - RESPONSÁVEL PELA VALIDAÇÃO - AUTOMÁTICO', 'Processo - RESPONSÁVEL PELA VALIDAÇÃO - MANUAL', 'PARA QUAL STATUS DESEJA ALTERAR?', 'MOTIVO DA ALTERAÇÃO DE STATUS', 'APTO PARA ACORDO?', 'ILEGITIMIDADE PASSIVA', 'PROJETO RECORTE MASSA?', 'Processo - EVENTO - CÍVEL', 'Processo - POSSUI A INICIAL?', 'Processo - CARTEIRA ESTRATÉGICO - CIVEL', 'Processo - MOTIVO DA APTIDÃO ?', 'PROCESSO - VIA NO POLO', 'PROCESSO - OBJETO CÍVEL (2) - NEW - SECUNDÁRIO',  'FEIRÃO LIMPA NOME', 'DESENROLA BRASIL', 'SUBSÍDIOS DISPONIBILIZADOS ATÉ O MOMENTO DE INSERÇÃO DA DEFESA?', 'APRESENTADA DEFESA GENÉRICA?', 'CATEGORIZAÇÃO BANQI', 'GRUPO CASAS BAHIA NO POLO DA AÇÃO?', 'Processo - DATA DA ABERTURA DE CONTA', 'PAGAR HONORÁRIOS AO ESCRITÓRIO?', 'ADVOGADO OFENSOR (CÍV. MASSA)', 'CÍVEL MASSA - FLUXO DE RESSARCIMENTO?','INDIQUE A FILIAL'])

In [0]:
# Assume que 'df' é seu DataFrame
df_final.columns = (df_final.columns
                .str.upper()                        # Minúsculo
                .str.normalize('NFKD')              # Normaliza acentos
                .str.encode('ascii', errors='ignore') # Remove caracteres não-ASCII
                .str.decode('utf-8')                # Volta para string
                .str.replace(r'[^\w\s]', '', regex=True) # Remove símbolos (ex: parenteses)
                .str.replace(r'\s+', '_', regex=True)    # Troca espaço por underscore
             )

In [0]:
df_final.columns

In [0]:
# 1. Defina a lista de colunas que você quer alterar
colunas_datas = ['DATA_REGISTRADO', 'DATA_DO_PEDIDO','DATA_DA_COMPRA','DATA_DE_REGISTRO_DO_ENCERRAMENTO']

# 2. Garante que elas sejam datetime primeiro (caso estejam como texto)
df_final[colunas_datas] = df_final[colunas_datas].apply(pd.to_datetime, errors='coerce')

# 3. Remove as horas (deixa apenas a data)
# Usamos lambda x: x.dt.date para aplicar a transformação em cada coluna individualmente
df_final[colunas_datas] = df_final[colunas_datas].apply(lambda x: x.dt.date)

In [0]:
df_final_estoque = df_final

In [0]:
df_final_estoque.dtypes

In [0]:
df_estoque_spark = spark.createDataFrame(df_final_estoque)

df_estoque_spark.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("juridico_comum.S_Civil_Canal_Estoque")

In [0]:
#Seleciona apenas colunas necessárias do
colunas_finais = [
    'PROCESSO - ID',
    'DATA REGISTRADO',
    'NÚMERO DO PEDIDO',
    'DATA DO PEDIDO', 
    'ORIGEM DA COMPRA',
    'FILIAL',
    'OBJETO CÍVEL (1) - PRINCIPAL',
    'COMPRA DIRETA OU MARKETPLACE? - NEW'
]

# O df passa a ter apenas essas colunas
df_final_24_25 = df_24_25[colunas_finais]

In [0]:
df_final_24_25.shape

In [0]:
# O errors='coerce' transforma o ano 3012 em NaT
df_final_24_25['DATA REGISTRADO'] = pd.to_datetime(df_final_24_25['DATA REGISTRADO'], errors='coerce')

# Aplica o filtro (os valores NaT são ignorados automaticamente nessa comparação)
df_final_24_25 = df_final_24_25[df_final_24_25['DATA REGISTRADO'] > '2024-01-01']

In [0]:
df_final_24_25.shape

In [0]:
# Assume que 'df' é seu DataFrame
df_final_24_25.columns = (df_final_24_25.columns
                .str.upper()                        # Minúsculo
                .str.normalize('NFKD')              # Normaliza acentos
                .str.encode('ascii', errors='ignore') # Remove caracteres não-ASCII
                .str.decode('utf-8')                # Volta para string
                .str.replace(r'[^\w\s]', '', regex=True) # Remove símbolos (ex: parenteses)
                .str.replace(r'\s+', '_', regex=True)    # Troca espaço por underscore
             )

In [0]:
#Cria a delta 
df_24_25_spark = spark.createDataFrame(df_final_24_25)

df_24_25_spark.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("juridico_comum.S_Civil_24_25")

In [0]:
colunas_finais_1 = ['DataProtocolo','NumeroPedido', 'TipoIncidente', 'CategoriaN1', 'CategoriaN2']

df_on = df_on[colunas_finais_1]

In [0]:
df_on.shape

In [0]:
# O errors='coerce' transforma o ano 3012 em NaT
df_on['DataProtocolo'] = pd.to_datetime(df_on['DataProtocolo'], errors='coerce')

# Aplica o filtro (os valores NaT são ignorados automaticamente nessa comparação)
df_on = df_on[df_on['DataProtocolo'] > '2024-01-01']

In [0]:
df_on.shape

In [0]:
# Assume que 'df' é seu DataFrame
df_on.columns = (df_on.columns
                .str.upper()                        # Minúsculo
                .str.normalize('NFKD')              # Normaliza acentos
                .str.encode('ascii', errors='ignore') # Remove caracteres não-ASCII
                .str.decode('utf-8')                # Volta para string
                .str.replace(r'[^\w\s]', '', regex=True) # Remove símbolos (ex: parenteses)
                .str.replace(r'\s+', '_', regex=True)    # Troca espaço por underscore
             )

In [0]:
# Force the column to be consistently strings
df_on['DATAPROTOCOLO'] = df_on['DATAPROTOCOLO'].astype(str)

# Handle cases where Pandas converted actual nulls to the string "nan" or "None"
df_on['DATAPROTOCOLO'] = df_on['DATAPROTOCOLO'].replace({'nan': None, 'None': None})

# Now convert to Spark
df_on_spark = spark.createDataFrame(df_on)

df_on_spark.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("juridico_comum.S_Canal_on")

In [0]:
colunas_finais_off = ['DataProtocolo',  'TipoInicial', 'NumeroPedido','CategoriaN1', 'CategoriaN2']

df_off = df_off[colunas_finais_off]

In [0]:
df_off.shape

In [0]:
# O errors='coerce' transforma o ano 3012 em NaT
df_off['DataProtocolo'] = pd.to_datetime(df_off['DataProtocolo'], errors='coerce')

# Aplica o filtro (os valores NaT são ignorados automaticamente nessa comparação)
df_off = df_off[df_off['DataProtocolo'] > '2024-01-01']

In [0]:
df_off.shape

In [0]:
# Assume que 'df' é seu DataFrame
df_off.columns = (df_off.columns
                .str.upper()                        # Minúsculo
                .str.normalize('NFKD')              # Normaliza acentos
                .str.encode('ascii', errors='ignore') # Remove caracteres não-ASCII
                .str.decode('utf-8')                # Volta para string
                .str.replace(r'[^\w\s]', '', regex=True) # Remove símbolos (ex: parenteses)
                .str.replace(r'\s+', '_', regex=True)    # Troca espaço por underscore
             )

In [0]:
df_off.head()

In [0]:
# Force the column to be consistently strings
df_off['DATAPROTOCOLO'] = df_off['DATAPROTOCOLO'].astype(str)

# Handle cases where Pandas converted actual nulls to the string "nan" or "None"
df_off['DATAPROTOCOLO'] = df_off['DATAPROTOCOLO'].replace({'nan': None, 'None': None})

# Now convert to Spark
df_off_spark = spark.createDataFrame(df_off)

df_off_spark.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("juridico_comum.S_Canal_off")